<a href="https://colab.research.google.com/github/kuds/rl-drone/blob/main/%5BDrone%20Racer%5D%20Soft%20Actor-Critic%20(SAC).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Drone Track] Soft Actor-Critic (SAC)


In [ ]:
!pip install mujoco

# Set up GPU rendering.
from google.colab import files
import distutils.util
import os
import subprocess
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

# Check if installation was succesful.
try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Other imports and helper functions
import time
import itertools
import numpy as np

# Graphics and plotting.
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

from IPython.display import clear_output
clear_output()

In [ ]:
!pip install robot_descriptions
!pip install gymnasium
!pip install stable-baselines3

In [ ]:
# --- Standard Library ---
import csv
import json
import math
import os
import platform
import shutil
import xml.etree.ElementTree as ET
from datetime import datetime
from importlib.metadata import version

# --- Third-Party: Data Science & Math ---
import matplotlib
import matplotlib.pyplot as plt
import numpy
import numpy as np
import torch
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import BSpline, make_interp_spline, splev, splprep
from scipy.optimize import minimize, minimize_scalar

# --- Third-Party: RL, Robotics & Physics ---
import gymnasium
import mujoco
import robot_descriptions
from gymnasium import utils
from gymnasium.envs.mujoco import MujocoEnv
from gymnasium.spaces import Box
from robot_descriptions import cf2_mj_description
from robot_descriptions.cf2_mj_description import MJCF_PATH as CF2_PATH

# --- Stable Baselines3 ---
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import (
    BaseCallback,
    CallbackList,
    CheckpointCallback,
    EvalCallback,
)
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_atari_env, make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import (
    DummyVecEnv,
    VecNormalize,
    VecVideoRecorder,
)

# --- Environment Specific (Colab) ---
try:
    import google.colab.drive
except ImportError:
    pass # Gracefully handle if not running in Colab

In [ ]:
print(f"Python Version: {platform.python_version()}")
print(f"Gymnasium Version: {version('gymnasium')}")
print(f"Robot Description Version: {version('robot_descriptions')}")
print(f"Numpy Version: {version('numpy')}")
print(f"Mujoco Version: {version('mujoco')}")
print(f"Stable-Baselines3 Version: {version('stable-baselines3')}")
print(f"Matplotlib Version: {version('matplotlib')}")
print(f"Torch Version: {version('torch')}")
print(f"Is Cuda Available: {torch.cuda.is_available()}")
print(f"Cuda Version: {torch.version.cuda}")
if torch.cuda.is_available(): print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
rl_type = "SAC"
env_str = "DroneRacer-v0"
name_prefix = f"{env_str}_{rl_type}".lower()
log_dir = ""
use_google_drive = True
if use_google_drive:
    parent_path = "/content/gdrive"
    google.colab.drive.mount(parent_path, force_remount=True)
    log_dir = "{}/MyDrive/Finding Theta/logs/{}/{}".format(parent_path,
                                                       env_str,
                                                       rl_type)
else:
    log_dir = "/content/logs/{}/{}".format(env_str, rl_type)

In [ ]:
training_data_path = os.path.join(log_dir, "training jobs")
time_folder = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
model_folder_path = os.path.join(log_dir, "training jobs", time_folder)

In [ ]:
notebook_config = {
    "rl_type": rl_type,
    "env_str": env_str,
    "name_prefix": name_prefix,
    "video_playback": 5,
    "log_dir": log_dir,
    "model_folder_path": model_folder_path,
    "time_folder": time_folder,
    "training_data_path": training_data_path,
    "use_google_drive": use_google_drive,
    "python_version": platform.python_version(),
    "torch_version": version('torch'),
    "cuda_device_name": torch.cuda.get_device_name(),
    "is_cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gymnasium_version": version('gymnasium'),
    "robot_descriptions_version": version('robot_descriptions'),
    "stable_baselines3_version": version('stable-baselines3'),
    "numpy_version": version('numpy'),
    "mujoco_version": version('mujoco'),
}

env_config = {
    "frame_stack": 4,
    "episode_length": 5_000,
    "sphere_size": 0.1,
    "track_size": 1,
    "number_of_checkpoints": 12,
    "reward_function": "multiplicative_inverse",
    "terminate_without_contact": 450,
    "speed_factor": 0.25,
    "track_height": 0.25,
    "max_distance": 0.5,
    "time_penalty": 0.1,
    "out_of_bounds_penalty": -100,
    "no_contact_penalty": -100,
    "complete_bonus": 250,
    "contact_bonus": 25
}

model_config = {
    "eval_freq": 25_000,
    "n_envs": 4,
    "total_timesteps": 10_000_000,
    "n_eval_episodes": 25
}

In [ ]:
#Create Folders
os.makedirs(log_dir, exist_ok=True)
os.makedirs(training_data_path, exist_ok=True)
os.makedirs(model_folder_path, exist_ok=True)

In [ ]:
with open(os.path.join(model_folder_path, 'config.json'), 'w') as fp:
    json.dump({
        "notebook_config": notebook_config,
        "env_config": env_config,
        "model_config": model_config
    }, fp, indent=4)

In [ ]:
csv_header = ["step",
              "drone_pos_x",
              "drone_pos_y",
              "drone_pos_z",
              "target_pos_x",
              "target_pos_y",
              "target_pos_z",
              "reward",
              "reward_normalized",
              "total_reward",
              "touch_reported",
              "sensor_reading",
              "made_contact",
              "laps",
              "distance_to_target",
              "drone_speed",
              "steps_without_contact",
              "total_contacts",
              "done"]

In [ ]:
class ReformatEvalCallback(BaseCallback):
    def __init__(
        self,
        save_path: str,
        eval_file: str,
        save_freq: int = 5_000,
        csv_file_name: str ="rl_model",
        verbose: int = 0):

        super().__init__(verbose)
        self.eval_file = eval_file
        self.csv_file_name = csv_file_name
        self.save_freq = save_freq
        self.save_path = save_path
        # Those variables will be accessible in the callback
        # (they are defined in the base class)
        # The RL model
        # self.model = None  # type: BaseAlgorithm
        # An alias for self.model.get_env(), the environment used for training
        # self.training_env # type: VecEnv
        # Number of time the callback was called
        # self.n_calls = 0  # type: int
        # num_timesteps = n_envs * n times env.step() was called
        # self.num_timesteps = 0  # type: int
        # local and global variables
        # self.locals = {}  # type: Dict[str, Any]
        # self.globals = {}  # type: Dict[str, Any]
        # The logger object, used to report things in the terminal
        # self.logger # type: stable_baselines3.common.logger.Logger
        # Sometimes, for event callback, it is useful
        # to have access to the parent object
        # self.parent = None  # type: Optional[BaseCallback]

    def _on_step(self) -> bool:
        if self.n_calls % self.save_freq == 0:
            """
            Loads data from a .npz file, calculates the average and standard
            deviation for each row, and saves the results to a CSV file.
            """
            try:
                # Load the .npz file. A .npz file is a dictionary-like object
                # that stores arrays.
                with numpy.load(self.eval_file) as data_archive:
                    # .npz files can contain multiple arrays. We'll assume the user
                    # wants to process the first array in the archive.
                    # The 'files' attribute lists the names of the arrays stored.
                    if not data_archive.files:
                        print(f"Error: No arrays found in {self.eval_file}")
                        return

                    # Timesteps
                    timesteps = data_archive['timesteps']

                    # Eposide Length
                    ep_lengths = data_archive['ep_lengths']

                    # Calculate the average (mean) for each row (axis=1).
                    row_averages_ep = numpy.mean(ep_lengths, axis=1)

                    # Calculate the standard deviation for each row (axis=1).
                    row_std_devs_ep = numpy.std(ep_lengths, axis=1)

                    # Results
                    results = data_archive['results']

                    # Calculate the average (mean) for each row (axis=1).
                    row_averages_reward = numpy.mean(results, axis=1)

                    # Calculate the standard deviation for each row (axis=1).
                    row_std_devs_reward = numpy.std(results, axis=1)

                    # Write the calculated statistics to the specified CSV file.
                    csv_file_path = os.path.join(self.save_path, f"{self.csv_file_name}.csv")
                    with open(csv_file_path, 'w', newline='') as csvfile:
                        # Create a CSV writer object.
                        csv_writer = csv.writer(csvfile)

                        # Write the header row.
                        csv_writer.writerow(['steps', 'reward_average', 'reward_standard_deviation', 'episode_average', 'episode_standard_deviation'])

                        # Zip the averages and standard deviations together to iterate
                        # over them in pairs.
                        for step, res_avg, res_std, ep_avg, ep_std in zip(timesteps, row_averages_reward, row_std_devs_reward, row_averages_ep, row_std_devs_ep):
                            csv_writer.writerow([step, res_avg, res_std, ep_avg, ep_std])

                    # print(f"Successfully created CSV file at: {csv_file_path}")

            except FileNotFoundError:
                print(f"Error: The file at {self.eval_file} was not found.")
            except Exception as e:
                print(f"An unexpected error occurred: {e}")
        return True


In [ ]:
class VecNormalizeSaveCallback(BaseCallback):
    def __init__(
        self,
        save_path: str,
        file_name: str = "best_model_vec_normalize.pkl",
        verbose: int = 0):

        super().__init__(verbose)
        self.file_name = file_name
        self.save_path = save_path

    def _on_step(self) -> bool:
        self.training_env.save(os.path.join(self.save_path, self.file_name))
        return True


In [ ]:
# --- Configuration ---
# The script will modify this file directly
filename = cf2_mj_description.MJCF_PATH

# 1. Parse the existing XML file
try:
    tree = ET.parse(filename)
    root = tree.getroot()
except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found.")
    exit()
except ET.ParseError:
    print(f"Error: The file '{filename}' is not a valid XML file.")
    exit()

# 2. Find both the <worldbody> and <sensor> parent nodes
worldbody_node = root.find('worldbody')
sensor_node = root.find('sensor')

# 3. Validate that both parent nodes were found
if worldbody_node is not None and sensor_node is not None:
    changes_made = False
    print("Found <worldbody> and <sensor> nodes. Checking for elements...")

    # --- Action 1: Check for and add the fly_zone ---
    # The XPath "./body[@name='fly_zone']" finds a body element with a specific name
    if worldbody_node.find("./site[@name='fly_sensor']") is None:
        print(" -> Body 'fly_zone' not found. Adding it...")
        site_attributes = {
            'name': 'fly_sensor',
            'pos': "0 0 " + str(env_config["track_height"]),
            'size': str(env_config["sphere_size"]),
            'type': 'sphere',
            'rgba': '0 1 0 .25'
        }
        # ET.SubElement(ET.Element('site', attrib=site_attributes))
        ET.SubElement(worldbody_node, 'site', attrib=site_attributes)
        changes_made = True
    else:
        print(" -> Body 'fly_zone' already exists. Skipping.")

    # --- Action 2: Check for and add the touch_sensor ---
    if sensor_node.find("./touch[@name='touch_sensor']") is None:
        print(" -> Sensor 'touch_sensor' not found. Adding it...")
        touch_attributes = {'name': 'touch_sensor', 'site': 'fly_sensor'}
        ET.SubElement(sensor_node, 'touch', attrib=touch_attributes)
        changes_made = True
    else:
        print(" -> Sensor 'touch_sensor' already exists. Skipping.")

    # 4. Write to the file ONLY if changes were made
    if changes_made:
        ET.indent(tree, space="  ", level=0)
        tree.write(filename, encoding='utf-8', xml_declaration=True)
        print(f"\n✅ Successfully applied updates to '{filename}'.")
    else:
        print(f"\n✅ No updates needed. All elements already exist in '{filename}'.")

else:
    # This block runs if one or both parent nodes are missing
    print("❌ Error: Could not find both <worldbody> and <sensor> nodes. No changes were made.")
    if worldbody_node is None:
        print(" -> Missing <worldbody> node.")
    if sensor_node is None:
        print(" -> Missing <sensor> node.")

In [ ]:
def modified_tanh(x):
    """
    Calculates the modified tanh function.
    """
    k = 5.406
    x0 = 0.575
    return 0.5 * (1 - numpy.tanh(k * (x - x0)))

def modified_tanh_final(x, factor=0.5):
    """
    Calculates the final modified tanh function.
    """
    k = 2.941
    x0 = 1.1
    return 0.5 * (1 - numpy.tanh(k * (x - x0)))

def multiplicative_inverse(x, factor=2):
    """
    Calculates the final modified tanh function.
    """
    return 1.0 / (1.0 + x * factor)

def round_tuple_elements(numeric_tuple):
    """
    Rounds each element of a numeric tuple to two decimal places.

    Args:
        numeric_tuple: A tuple of numbers (integers or floats).

    Returns:
        A new tuple with each element rounded to 2 decimal places.
    """
    if not isinstance(numeric_tuple, tuple):
        raise TypeError("Input must be a tuple.")

    # Use a generator expression within the tuple() constructor for efficiency
    rounded_tuple = tuple(round(item, 2) for item in numeric_tuple)
    return rounded_tuple

In [ ]:
def generate_equidistant_points(radius: float, num_points: int, z_height: int, add_noise: bool=False) -> list[tuple[float, float, float]]:
    """
    Generates a list of points that are equally spaced around a circle.

    The points are generated starting from the positive x-axis (angle 0) and
    moving counter-clockwise.

    Args:
        radius: The radius of the circle.
        num_points: The total number of points to generate on the circle's circumference.

    Returns:
        A list of (x, y, z) tuples representing the coordinates of the points.
    """
    if radius <= 0:
        raise ValueError("Radius must be a positive number.")
    if num_points <= 0:
        raise ValueError("Number of points must be a positive integer.")

    points = []
    # The angle between each point in radians
    angle_step = 2 * math.pi / num_points

    for i in range(num_points):
        # Calculate the angle for the current point
        angle = angle_step * i
        # Convert from polar coordinates (radius, angle) to Cartesian (x, y)
        x = radius * math.cos(angle) + radius
        y = radius * math.sin(angle)
        # Append the point with a z-coordinate of 0
        points.append((x, y, z_height))

    return points

def get_next_clockwise_point(current_point: tuple[float, float, float], all_points: list[tuple[float, float, float]]) -> tuple[float, float, float]:
    """
    Finds the next point in a list of points in a clockwise direction.

    It finds the index of the current_point in the list and returns the
    point at the previous index, wrapping around if necessary.

    Args:
        current_point: The (x, y, z) tuple of the starting point.
        all_points: A list of all equidistant points on the circle.

    Returns:
        The (x, y, z) tuple of the next point in the clockwise direction.

    Raises:
        ValueError: If the current_point is not found in the list of all_points.
    """
    try:
        # Find the index of the current point. We use np.isclose to handle
        # potential floating point inaccuracies.
        idx = -1
        for i, p in enumerate(all_points):
            # print(p)
            if np.isclose(p[0], current_point[0]) and np.isclose(p[1], current_point[1]):
                idx = i
                break

        if idx == -1:
            raise ValueError("Point not found")

    except ValueError:
        raise ValueError(f"The provided current_point is not in the list of points. Point: {current_point}")

    num_points = len(all_points)
    # The next index in a clockwise direction is the previous index in the list.
    # The math (idx - 1 + num_points) % num_points correctly handles the wrap-around
    # from index 0 to the last index.
    next_index = (idx - 1 + num_points) % num_points

    return all_points[next_index]

def add_polar_noise_to_points(points, angle_noise_std_dev, z_noise_std_dev):
    """
    Applies noise to a list of 3D points.
    - Z-axis noise is added directly.
    - X/Y noise is added to the *angle* in the polar coordinate space.

    Args:
        points (list or np.array): The input (N, 3) coordinates.
        angle_noise_std_dev (float): Standard deviation of the noise for the
                                     angle (in radians).
        z_noise_std_dev (float): Standard deviation of the noise for the Z-axis.

    Returns:
        list: A new list of (x, y, z) tuples with noise applied.
    """
    points_arr = np.array(points)
    num_points = len(points_arr)

    # 1. Generate Z-axis noise
    # np.random.normal(mean, std_dev, number_of_samples)
    z_noise = np.random.normal(0, z_noise_std_dev, num_points)
    new_z = points_arr[:, 2] + z_noise

    # 2. Generate Angle noise (in radians)
    angle_noise = np.random.normal(0, angle_noise_std_dev, num_points)

    # 3. Apply X/Y (polar) noise
    x = points_arr[:, 0]
    y = points_arr[:, 1]

    # Convert to polar coordinates (r, theta)
    # np.hypot is a stable way to calculate sqrt(x^2 + y^2)
    r = np.hypot(x, y)
    # np.arctan2 correctly handles all quadrants for the angle
    theta = np.arctan2(y, x)

    # Add noise to the angle
    new_theta = theta + angle_noise

    # Convert back to Cartesian coordinates
    # The radius 'r' remains unchanged, as requested
    new_x = r * np.cos(new_theta)
    new_y = r * np.sin(new_theta)

    # 4. Assemble the new noisy points
    new_points_arr = np.stack((new_x, new_y, new_z), axis=1)

    # Convert back to a list of tuples for consistency
    new_points_list = [tuple(row) for row in new_points_arr]

    return new_points_list

def add_radial_noise_to_points(points, r_noise_std_dev, z_noise_std_dev, skip_origin=False):
    """
    Applies noise to a list of 3D points.
    - Z-axis noise is added directly.
    - X/Y noise is added to the *radius* (distance from center) in polar space,
      preserving the original angle.

    Args:
        points (list or np.array): The input (N, 3) coordinates.
        r_noise_std_dev (float): Standard deviation of the noise for the radius.
        z_noise_std_dev (float): Standard deviation of the noise for the Z-axis.

    Returns:
        list: A new list of (x, y, z) tuples with noise applied.
    """

    points_arr = np.array(points)
    orginal_points = points_arr.copy()
    origin_index = 0
    #print(points_arr)
    if skip_origin:
        # Apply the inverse of the mask to filter the data
        # The '~' symbol inverts the mask (True becomes False)
        is_x_zero = np.isclose(points_arr[:, 0], 0, atol=1e-05)
        is_y_zero = np.isclose(points_arr[:, 1], 0, atol=1e-05)

        # 3. Combine them (We want to remove where BOTH are zero)
        mask = is_x_zero & is_y_zero
        origin_index = np.argmax(mask)
        #print(mask)
        #print(origin_index)
        #print(points_arr[origin_index])
        #print(origin_index)
        #print(type(points_arr))
        #print(type(points_arr[0]))
        # for i in range(len(points_arr)):
        #     print(points_arr[i])
        #mask = (points_arr[:, 0] == 0.0) & (points_arr[:, 1] == 0.0)
        #print(mask)
        points_arr = points_arr[~mask]
        #print(f"Filtered {mask.sum()} points at origin.")

    num_points = len(points_arr)

    # 1. Generate Z-axis noise
    z_noise = np.random.normal(0, z_noise_std_dev, num_points)
    new_z = points_arr[:, 2] + z_noise

    # 2. Generate Radius noise
    r_noise = np.random.normal(0, r_noise_std_dev, num_points)

    # 3. Apply X/Y (radial) noise
    x = points_arr[:, 0]
    y = points_arr[:, 1]

    # Convert to polar coordinates (r, theta)
    r = np.hypot(x, y)
    theta = np.arctan2(y, x)

    # Add noise to the radius (instead of the angle)
    new_r = r + r_noise

    # Note: If r + noise becomes negative, the point effectively reflects
    # across the origin, which is mathematically valid in this context.

    # Convert back to Cartesian coordinates using the ORIGINAL theta
    new_x = new_r * np.cos(theta)
    new_y = new_r * np.sin(theta)

    # 4. Assemble the new noisy points
    new_points_arr = np.stack((new_x, new_y, new_z), axis=1)
    #print(new_points_arr)
    if skip_origin:
        new_points_arr = np.insert(new_points_arr, origin_index, orginal_points[origin_index], axis=0)

    # Convert back to a list of tuples
    new_points_list = [tuple(row) for row in new_points_arr]
    #print(new_points_list)

    return new_points_list

def add_radial_noise_to_points_rng(points, r_noise_std_dev, z_noise_std_dev, skip_origin=False, seed=None):
    """
    Applies noise to a list of 3D points with a specific seed for repeatability.

    Args:
        points (list or np.array): The input (N, 3) coordinates.
        r_noise_std_dev (float): Standard deviation of the noise for the radius.
        z_noise_std_dev (float): Standard deviation of the noise for the Z-axis.
        seed (int, optional): A seed value for the random number generator.
                              If None, the system (clock) time is used.

    Returns:
        list: A new list of (x, y, z) tuples with noise applied.
    """
    # Create a dedicated random number generator (RNG) instance.
    # This ensures that setting the seed here doesn't affect random numbers
    # elsewhere in your program (unlike np.random.seed()).
    rng = np.random.default_rng(seed)

    points_arr = np.array(points)
    num_points = len(points_arr)
    orginal_points = points_arr.copy()
    origin_index = 0
    if skip_origin:
        # Apply the inverse of the mask to filter the data
        # The '~' symbol inverts the mask (True becomes False)
        is_x_zero = np.isclose(points_arr[:, 0], 0, atol=1e-05)
        is_y_zero = np.isclose(points_arr[:, 1], 0, atol=1e-05)

        # 3. Combine them (We want to remove where BOTH are zero)
        mask = is_x_zero & is_y_zero
        origin_index = np.argmax(mask)
        points_arr = points_arr[~mask]

    num_points = len(points_arr)

    # 1. Generate Z-axis noise using the rng instance
    z_noise = rng.normal(0, z_noise_std_dev, num_points)
    new_z = points_arr[:, 2] + z_noise

    # 2. Generate Radius noise using the rng instance
    r_noise = rng.normal(0, r_noise_std_dev, num_points)

    # 3. Apply Radial noise
    x = points_arr[:, 0]
    y = points_arr[:, 1]

    # Convert to polar coordinates
    r = np.hypot(x, y)
    theta = np.arctan2(y, x)

    # Add noise to the radius
    new_r = r + r_noise

    # Convert back to Cartesian coordinates
    new_x = new_r * np.cos(theta)
    new_y = new_r * np.sin(theta)

    # 4. Assemble
    new_points_arr = np.stack((new_x, new_y, new_z), axis=1)
    if skip_origin:
        new_points_arr = np.insert(new_points_arr,
                                   origin_index,
                                   orginal_points[origin_index],
                                   axis=0)

    return [tuple(row) for row in new_points_arr]

In [ ]:
def generate_equidistant_points(radius: float, num_points: int, z_height: int, add_noise: bool=False) -> list[tuple[float, float, float]]:
    """
    Generates a list of points that are equally spaced around a circle.

    The points are generated starting from the positive x-axis (angle 0) and
    moving counter-clockwise.

    Args:
        radius: The radius of the circle.
        num_points: The total number of points to generate on the circle's circumference.

    Returns:
        A list of (x, y, z) tuples representing the coordinates of the points.
    """
    if radius <= 0:
        raise ValueError("Radius must be a positive number.")
    if num_points <= 0:
        raise ValueError("Number of points must be a positive integer.")

    points = []
    # The angle between each point in radians
    angle_step = 2 * math.pi / num_points

    for i in range(num_points):
        # Calculate the angle for the current point
        angle = angle_step * i
        # Convert from polar coordinates (radius, angle) to Cartesian (x, y)
        x = radius * math.cos(angle) + radius
        y = radius * math.sin(angle)
        # Append the point with a z-coordinate of 0
        points.append((x, y, z_height))

    return points

def get_next_clockwise_point(current_point: tuple[float, float, float], all_points: list[tuple[float, float, float]]) -> tuple[float, float, float]:
    """
    Finds the next point in a list of points in a clockwise direction.

    It finds the index of the current_point in the list and returns the
    point at the previous index, wrapping around if necessary.

    Args:
        current_point: The (x, y, z) tuple of the starting point.
        all_points: A list of all equidistant points on the circle.

    Returns:
        The (x, y, z) tuple of the next point in the clockwise direction.

    Raises:
        ValueError: If the current_point is not found in the list of all_points.
    """
    try:
        # Find the index of the current point. We use np.isclose to handle
        # potential floating point inaccuracies.
        idx = -1
        for i, p in enumerate(all_points):
            if np.isclose(p[0], current_point[0]) and np.isclose(p[1], current_point[1]):
                idx = i
                break

        if idx == -1:
            raise ValueError("Point not found")

    except ValueError:
        raise ValueError("The provided current_point is not in the list of points.")

    num_points = len(all_points)
    # The next index in a clockwise direction is the previous index in the list.
    # The math (idx - 1 + num_points) % num_points correctly handles the wrap-around
    # from index 0 to the last index.
    next_index = (idx - 1 + num_points) % num_points

    return all_points[next_index]

# --- Example Usage and Plotting ---
if __name__ == "__main__":
    # --- Configuration ---
    CIRCLE_RADIUS = env_config["track_size"]
    NUM_POINTS = env_config["number_of_checkpoints"]
    TRACK_HEIGHT = env_config["track_height"]

    # --- Generation ---
    # 1. Generate the 15 equidistant points
    equidistant_points = generate_equidistant_points(CIRCLE_RADIUS, NUM_POINTS, TRACK_HEIGHT)

    print(f"Generated {NUM_POINTS} equidistant points on a circle with radius {CIRCLE_RADIUS}:")
    for i, p in enumerate(equidistant_points):
        print(f"  Point {i:02d}: (X={p[0]:.2f}, Y={p[1]:.2f})")

    # --- Calculation ---
    # 2. Pick a starting point and find the next one clockwise
    # Let's start at point 3 for this example
    start_point_index = 3
    current_position = equidistant_points[start_point_index]

    next_position = get_next_clockwise_point(current_position, equidistant_points)

    print(f"\nStarting at point {start_point_index}: ({current_position[0]:.2f}, {current_position[1]:.2f})")
    print(f"The next point clockwise is: ({next_position[0]:.2f}, {next_position[1]:.2f})")

    # --- Plotting ---
    # 3. Visualize the points and the movement
    x_coords = [p[0] for p in equidistant_points]
    y_coords = [p[1] for p in equidistant_points]

    plt.figure(figsize=(8, 8))
    # Plot all the points
    plt.scatter(x_coords, y_coords, label=f'{NUM_POINTS} Equidistant Points')
    # Highlight the starting point
    plt.scatter(current_position[0], current_position[1], color='green', s=150, zorder=5, label='Start Point')
    # Highlight the next point
    plt.scatter(next_position[0], next_position[1], color='red', s=150, zorder=5, label='Next Point (Clockwise)')

    # Add an arrow to show the direction
    plt.annotate('', xy=(next_position[0], next_position[1]), xytext=(current_position[0], current_position[1]),
                 arrowprops=dict(facecolor='black', shrink=0.1, width=1, headwidth=8))

    plt.title("Equidistant Points on a Circle")
    plt.xlabel("X Coordinate")
    plt.ylabel("Y Coordinate")
    plt.grid(True)
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.legend()
    plt.show()

In [ ]:
"""
3D Curve Fitting Script
Class-based implementation for creating smooth 3D curves through points.
Creates closed curves and provides nearest point queries.
"""


class Curve3D:
    """
    A class for creating and working with smooth 3D curves through points.

    Attributes:
    -----------
    original_points : ndarray
        The original input points
    splines : list of BSpline objects
        Spline representations for x, y, z dimensions
    degree : int
        Degree of the spline
    closed : bool
        Whether the curve is closed
    curve_points : ndarray
        Cached curve points for visualization
    """

    def __init__(self, points, degree=3, closed=True, num_samples=100):
        """
        Initialize a 3D curve through the given points.

        Parameters:
        -----------
        points : array-like, shape (n, 3)
            List of 3D points [(x1,y1,z1), (x2,y2,z2), ...]
        degree : int, optional (default=3)
            Degree of the spline (1=linear, 2=quadratic, 3=cubic)
        closed : bool, optional (default=True)
            If True, creates a closed curve connecting last point to first point
        num_samples : int, optional (default=100)
            Number of points to generate for visualization
        """
        self.original_points = np.array(points)
        self.degree = degree
        self.closed = closed

        if self.original_points.shape[0] < 2:
            raise ValueError("At least 2 points are required")

        if self.original_points.shape[1] != 3:
            raise ValueError("Points must be 3-dimensional")

        # Build the spline representation
        self._build_splines()

        # Generate curve points for visualization (including original points)
        # For closed curves, exclude the duplicate first point at the end
        if self.closed:
            original_t = self.t_params[:-1]
        else:
            original_t = self.t_params

        # Generate uniform samples
        t_uniform = np.linspace(0, 1, num_samples)

        # Combine and sort all t values, removing duplicates
        t_combined = np.unique(np.concatenate([original_t, t_uniform]))

        # Evaluate at all t values
        self.curve_points = self.evaluate(t_combined)

    def _build_splines(self):
        """Build the spline representation of the curve."""
        points = self.original_points.copy()

        # For closed curves, append the first point at the end
        if self.closed:
            points = np.vstack([points, points[0:1]])

        n_points = len(points)

        # Adjust degree if necessary
        degree = min(self.degree, n_points - 1)

        # Create parameter values (arc length parameterization)
        distances = np.sqrt(np.sum(np.diff(points, axis=0)**2, axis=1))
        cumulative_distances = np.concatenate([[0], np.cumsum(distances)])
        self.t_params = cumulative_distances / cumulative_distances[-1]

        # Create splines for each dimension
        self.splines = []
        for dim in range(3):
            # For closed curves, use periodic boundary conditions
            if self.closed and n_points > degree + 1:
                bc_type = 'periodic'
            else:
                bc_type = 'not-a-knot'

            try:
                spline = make_interp_spline(self.t_params, points[:, dim],
                                           k=degree, bc_type=bc_type)
            except ValueError:
                # If periodic fails, fall back to not-a-knot
                spline = make_interp_spline(self.t_params, points[:, dim],
                                           k=degree, bc_type='not-a-knot')

            self.splines.append(spline)

    def evaluate(self, t):
        """
        Evaluate the curve at parameter value(s) t.

        Parameters:
        -----------
        t : float or array-like
            Parameter value(s) in range [0, 1]

        Returns:
        --------
        points : ndarray, shape (3,) or (n, 3)
            Point(s) on the curve
        """
        t = np.atleast_1d(t)
        points = np.column_stack([spline(t) for spline in self.splines])
        return points.squeeze() if len(t) == 1 else points

    def find_nearest_point(self, query_point, num_samples=1000, refine=True):
        """
        Find the nearest point on the curve to a given query point.

        Parameters:
        -----------
        query_point : array-like, shape (3,)
            The query point in 3D space
        num_samples : int, optional (default=1000)
            Number of points to sample along the curve for initial search
        refine : bool, optional (default=True)
            If True, refine the result using optimization

        Returns:
        --------
        nearest_point : ndarray, shape (3,)
            The nearest point on the curve
        distance : float
            Distance from query point to nearest point
        t_nearest : float
            Parameter value of the nearest point
        """
        query_point = np.array(query_point)

        # Sample the curve densely
        t_samples = np.linspace(0, 1, num_samples)
        curve_samples = self.evaluate(t_samples)

        # Compute distances to all sampled points
        distances = np.linalg.norm(curve_samples - query_point, axis=1)

        # Find the closest sampled point
        min_idx = np.argmin(distances)
        t_initial = t_samples[min_idx]

        if not refine:
            nearest_point = curve_samples[min_idx]
            distance = distances[min_idx]
            return nearest_point, distance, t_initial

        # Refine using optimization
        def distance_function(t):
            """Function to minimize: distance from curve to query point."""
            t = np.clip(t, 0, 1)  # Ensure t stays in valid range
            point_on_curve = self.evaluate(t)
            return np.linalg.norm(point_on_curve - query_point)

        # Use a bounded optimization around the initial guess
        window = 0.1  # Search within ±10% of the curve
        bounds = (max(0, t_initial - window), min(1, t_initial + window))

        result = minimize_scalar(distance_function, bounds=bounds, method='bounded')

        t_nearest = result.x
        nearest_point = self.evaluate(t_nearest)
        distance = result.fun

        return nearest_point, distance, t_nearest

    def get_points(self, num_samples=None, include_original=True):
        """
        Get all points along the curve.

        Parameters:
        -----------
        num_samples : int, optional
            Number of points to generate along the curve.
            If None, returns the cached curve_points from initialization.
            If specified, generates new points with the given sampling.
        include_original : bool, optional (default=True)
            If True, ensures that the original input points are included
            in the returned set of points.

        Returns:
        --------
        points : ndarray, shape (num_samples, 3)
            Points along the curve, including original points if include_original=True
        """
        # If no num_samples specified, return cached points
        if num_samples is None:
            return self.curve_points.copy()

        if not include_original:
            # Simple uniform sampling without original points
            t_samples = np.linspace(0, 1, num_samples)
            return self.evaluate(t_samples)

        # Include original points by using their parameter values
        # For closed curves, exclude the duplicate first point at the end
        if self.closed:
            original_t = self.t_params[:-1]  # Exclude the appended first point
        else:
            original_t = self.t_params

        # Generate uniform samples
        t_uniform = np.linspace(0, 1, num_samples)

        # Combine and sort all t values, removing duplicates
        t_combined = np.unique(np.concatenate([original_t, t_uniform]))

        # Evaluate at all t values
        points = self.evaluate(t_combined)

        return points

    def get_length(self, num_samples=1000):
        """
        Estimate the total length of the curve.

        Parameters:
        -----------
        num_samples : int, optional (default=1000)
            Number of samples to use for length estimation

        Returns:
        --------
        length : float
            Approximate length of the curve
        """
        t_samples = np.linspace(0, 1, num_samples)
        curve_samples = self.evaluate(t_samples)

        # Compute distances between consecutive points
        segment_lengths = np.linalg.norm(np.diff(curve_samples, axis=0), axis=1)
        total_length = np.sum(segment_lengths)

        return total_length

    def visualize(self, highlight_points=None):
        """
        Visualize the curve and original points.

        Parameters:
        -----------
        highlight_points : list of tuples, optional
            List of (point, label, color) tuples to highlight on the plot
        """
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')

        # Plot original points
        ax.scatter(self.original_points[:, 0],
                   self.original_points[:, 1],
                   self.original_points[:, 2],
                   c='red', s=100, marker='o', label='Original Points', zorder=5)

        # Plot the curve
        ax.plot(self.curve_points[:, 0],
                self.curve_points[:, 1],
                self.curve_points[:, 2],
                'b-', linewidth=2, label='Fitted Curve')

        # Plot highlighted points if provided
        if highlight_points is not None:
            for point, label, color in highlight_points:
                ax.scatter(*point, c=color, s=150, marker='*',
                          label=label, zorder=10, edgecolors='black', linewidths=1)

        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_title('3D Curve' + (' (Closed)' if self.closed else ' (Open)'))
        ax.legend()

        plt.show()

        return fig, ax


# Example usage
if __name__ == "__main__":
    print("=" * 70)
    print("Example 1: Creating a Closed Curve and Finding Nearest Points")
    print("=" * 70)

    # Create a circular-ish closed curve
    theta = np.linspace(0, 2*np.pi, 8, endpoint=False)
    points = np.column_stack([
        5 * np.cos(theta),
        5 * np.sin(theta),
        np.linspace(0, 2, 8)
    ])

    # Create the curve
    curve = Curve3D(points, closed=True, num_samples=200)

    print(f"\nCreated curve through {len(points)} points")
    print(f"Curve length: {curve.get_length():.4f}")
    print(f"Curve is closed: {curve.closed}")

    # Test nearest point queries
    print("\n" + "-" * 70)
    print("Testing Nearest Point Queries:")
    print("-" * 70)

    # Query point 1: Near the curve
    query1 = np.array([6.0, 1.0, 1.0])
    nearest1, dist1, t1 = curve.find_nearest_point(query1)
    print(f"\nQuery Point 1: {query1}")
    print(f"Nearest Point on Curve: {nearest1}")
    print(f"Distance: {dist1:.4f}")
    print(f"Parameter t: {t1:.4f}")

    # Query point 2: Far from the curve
    query2 = np.array([0.0, 0.0, 10.0])
    nearest2, dist2, t2 = curve.find_nearest_point(query2)
    print(f"\nQuery Point 2: {query2}")
    print(f"Nearest Point on Curve: {nearest2}")
    print(f"Distance: {dist2:.4f}")
    print(f"Parameter t: {t2:.4f}")

    # Query point 3: Inside the curve
    query3 = np.array([0.0, 0.0, 1.0])
    nearest3, dist3, t3 = curve.find_nearest_point(query3)
    print(f"\nQuery Point 3: {query3}")
    print(f"Nearest Point on Curve: {nearest3}")
    print(f"Distance: {dist3:.4f}")
    print(f"Parameter t: {t3:.4f}")

    # Visualize
    print("\n" + "-" * 70)
    print("Generating visualization...")
    print("-" * 70)

    highlight = [
        (query1, 'Query 1', 'orange'),
        (nearest1, 'Nearest 1', 'green'),
        (query2, 'Query 2', 'purple'),
        (nearest2, 'Nearest 2', 'cyan'),
        (query3, 'Query 3', 'yellow'),
        (nearest3, 'Nearest 3', 'magenta')
    ]

    curve.visualize(highlight_points=highlight)

    print("\n" + "=" * 70)
    print("Example 2: Open Curve")
    print("=" * 70)

    # Create an open curve
    open_points = np.array([
        [0, 0, 0],
        [1, 1, 1],
        [2, 0.5, 2],
        [3, -0.5, 3],
        [4, 0, 4]
    ])

    open_curve = Curve3D(open_points, closed=False, num_samples=150)

    print(f"\nCreated open curve through {len(open_points)} points")
    print(f"Curve length: {open_curve.get_length():.4f}")

    # Find nearest point
    query_open = np.array([2.5, 2.0, 2.5])
    nearest_open, dist_open, t_open = open_curve.find_nearest_point(query_open)

    print(f"\nQuery Point: {query_open}")
    print(f"Nearest Point: {nearest_open}")
    print(f"Distance: {dist_open:.4f}")

    open_curve.visualize(highlight_points=[
        (query_open, 'Query', 'orange'),
        (nearest_open, 'Nearest', 'green')
    ])

    print("\n" + "=" * 70)
    print("Example 3: Performance Test - Multiple Queries")
    print("=" * 70)

    import time

    # Generate random query points
    np.random.seed(42)
    num_queries = 100
    random_queries = np.random.randn(num_queries, 3) * 3

    start_time = time.time()
    for query in random_queries:
        nearest, dist, t = curve.find_nearest_point(query)
    elapsed = time.time() - start_time

    print(f"\nProcessed {num_queries} nearest point queries")
    print(f"Total time: {elapsed:.4f} seconds")
    print(f"Average time per query: {elapsed/num_queries*1000:.2f} ms")

    print("\n" + "=" * 70)
    print("Example 4: Getting Points Along the Curve")
    print("=" * 70)

    # Get the cached curve points (from initialization)
    points_default = curve.get_points()
    print(f"\nDefault curve points: {len(points_default)} points")
    print(f"Shape: {points_default.shape}")

    # Generate curve points with custom sampling
    points_sparse = curve.get_points(num_samples=20)
    points_dense = curve.get_points(num_samples=500)

    print(f"\nSparse sampling: {len(points_sparse)} points")
    print(f"Dense sampling: {len(points_dense)} points")

    print(f"\nFirst few points (default):")
    for i in range(3):
        print(f"  Point {i}: [{points_default[i, 0]:7.4f}, {points_default[i, 1]:7.4f}, {points_default[i, 2]:7.4f}]")

    # Visualize different samplings
    fig = plt.figure(figsize=(15, 5))

    ax1 = fig.add_subplot(131, projection='3d')
    ax1.plot(points_sparse[:, 0], points_sparse[:, 1], points_sparse[:, 2],
             'b-o', linewidth=2, markersize=5)
    ax1.scatter(curve.original_points[:, 0], curve.original_points[:, 1],
                curve.original_points[:, 2], c='red', s=100, marker='o', zorder=5)
    ax1.set_title(f'Sparse ({len(points_sparse)} points)')
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')

    ax2 = fig.add_subplot(132, projection='3d')
    ax2.plot(points_default[:, 0], points_default[:, 1], points_default[:, 2],
             'g-', linewidth=2)
    ax2.scatter(curve.original_points[:, 0], curve.original_points[:, 1],
                curve.original_points[:, 2], c='red', s=100, marker='o', zorder=5)
    ax2.set_title(f'Default ({len(points_default)} points)')
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')

    ax3 = fig.add_subplot(133, projection='3d')
    ax3.plot(points_dense[:, 0], points_dense[:, 1], points_dense[:, 2],
             'purple', linewidth=2)
    ax3.scatter(curve.original_points[:, 0], curve.original_points[:, 1],
                curve.original_points[:, 2], c='red', s=100, marker='o', zorder=5)
    ax3.set_title(f'Dense ({len(points_dense)} points)')
    ax3.set_xlabel('X')
    ax3.set_ylabel('Y')
    ax3.set_zlabel('Z')

    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 70)
    print("Example 5: Evaluating the Curve at Custom Parameters")
    print("=" * 70)

    # Evaluate at specific parameter values
    t_values = [0.0, 0.25, 0.5, 0.75, 1.0]
    print("\nEvaluating curve at specific parameter values:")
    for t in t_values:
        point = curve.evaluate(t)
        print(f"t={t:.2f}: [{point[0]:7.4f}, {point[1]:7.4f}, {point[2]:7.4f}]")

In [ ]:
# 2. Define noise levels
z_noise_scale = 0.25  # Units of Z
angle_noise_scale = 0.25 # Radians (0.3 rad is ~17 degrees)

# --- Configuration ---
CIRCLE_RADIUS = env_config["track_size"]
NUM_POINTS = env_config["number_of_checkpoints"]
TRACK_HEIGHT = env_config["track_height"]

# --- Generation ---
# 1. Generate the 15 equidistant points
equidistant_points = generate_equidistant_points(CIRCLE_RADIUS, NUM_POINTS, TRACK_HEIGHT)

print(f"Generated {NUM_POINTS} equidistant points on a circle with radius {CIRCLE_RADIUS}:")
for i, p in enumerate(equidistant_points):
    print(f"  Point {i:02d}: (X={p[0]:.2f}, Y={p[1]:.2f})")

# 3. Create the new noisy points
noisy_coords = add_radial_noise_to_points_rng(equidistant_points, angle_noise_scale, z_noise_scale, skip_origin=True, seed=0)
print(f"Generated {NUM_POINTS} equidistant noisey points on a circle with radius {CIRCLE_RADIUS}:")
for i, p in enumerate(noisy_coords):
    print(f"  Point {i:02d}: (X={p[0]:.2f}, Y={p[1]:.2f}, Z={p[2]:.2f})")

# Create the curve
curve = Curve3D(equidistant_points, closed=True, num_samples=200)
noisy_curve = Curve3D(noisy_coords, closed=True, num_samples=200)

print(f"\nCreated curve through {len(equidistant_points)} points")
print(f"Curve length: {curve.get_length():.4f}")
print(f"Curve is closed: {curve.closed}")

# Visualize
print("\n" + "-" * 70)
print("Generating visualization...")
print("-" * 70)

noisy_curve.visualize()

In [ ]:
from typing import Dict
class DroneHoverEnv(MujocoEnv):
    """
    Custom Gymnasium environment for the drone hovering task.
    This version loads the drone model from `robot_descriptions`
    and programmatically builds the training scene.
    """

    metadata = {
    "render_modes": [
        "human",
        "rgb_array",
        "depth_array",
    ],
        "render_fps": 100,
    }

    def __init__(self, env_config: Dict, **kwargs):
        utils.EzPickle.__init__(self, **kwargs)
        for key, value in env_config.items():
            setattr(self, key, value)
        self.model_path = os.path.join(cf2_mj_description.PACKAGE_PATH, "scene.xml")
        self.total_reward = 0

        # --- 3. Define Spaces and Target (same as before) ---
        self.action_space = Box(
            low=np.array([0.0, -1.0, -1.0, -1.0]),
            high=np.array([0.35, 1.0, 1.0, 1.0]),
            dtype=np.float64
        )

        self.observation_space = Box(
            low=-np.inf, high=np.inf, shape=(19,), dtype=np.float64
        )

        # change shape of observation to your observation space size
        # load your MJCF model with env and choose frames count between actions
        MujocoEnv.__init__(
            self,
            self.model_path,
            frame_skip=5,
            observation_space=self.observation_space,
            **kwargs
        )

        self.base_points = generate_equidistant_points(self.track_size, self.number_of_checkpoints, self.track_height)
        self.track_points = add_radial_noise_to_points_rng(self.base_points, .25, .25, skip_origin=True, seed=0)
        self.current_point = (0,0, self.track_height)
        self.next_point = get_next_clockwise_point(self.current_point, self.track_points)

        # self.track_points = generate_equidistant_points(self.track_size, self.number_of_points, self.track_height)
        # self.current_point = (0,0, self.track_height)
        # self.next_point = get_next_clockwise_point(self.current_point, self.track_points)

        self.laps = 0
        self.total_contacts = 0
        self.steps_without_contact = 0
        self.noise_magnitude = 0.25
        self.step_number = 0
        self.target_pos_id = self.model.site("fly_sensor").id
        self.target_pos = self.model.site_pos[self.target_pos_id]
        self.drone_body_id = self.model.body('cf2').id
        self.gyro_sensor_id = self.model.sensor('body_gyro').id
        self.fly_sensor_id = self.model.sensor('touch_sensor').id
        self.original_site_pos = self.model.site_pos[self.target_pos_id].copy()

    def _update_track_position(self):
        """Updates the track position."""
        self.current_point = self.next_point
        self.model.site_pos[self.target_pos_id] = self.current_point
        self.next_point = get_next_clockwise_point(self.current_point, self.track_points)
        self.target_pos = self.model.site_pos[self.target_pos_id]

    def _get_obs(self):
        """Constructs the observation vector from the simulation data."""
        drone_pos = self.data.qpos[:3]
        drone_quat = self.data.qpos[3:7]
        drone_lin_vel = self.data.qvel[:3]
        drone_ang_vel = self.data.sensor(self.gyro_sensor_id).data
        vec_to_target = self.target_pos - drone_pos

        return np.concatenate([
            drone_pos,
            drone_quat,
            vec_to_target,
            drone_lin_vel,
            drone_ang_vel,
            self.target_pos
        ]).astype(np.float64)

    def step(self, action):
        """Applies an action, steps the simulation, and calculates the reward."""
        truncated = False
        terminated = False
        self.data.ctrl[:] = action
        for _ in range(self.frame_skip):
            mujoco.mj_step(self.model, self.data)

        self.step_number += 1
        drone_pos = self.data.qpos[:3]
        distance_to_target = np.linalg.norm(drone_pos - self.target_pos)
        # reward = modified_tanh_final(distance_to_target)

        reward = 0
        if self.reward_function == "multiplicative_inverse":
            reward = multiplicative_inverse(distance_to_target)
        elif self.reward_function == "modified_tanh":
            reward = modified_tanh(distance_to_target)
        elif self.reward_function == "modified_tanh_final":
            reward = modified_tanh_final(distance_to_target)
        elif self.reward_function == "none":
            pass
        else:
            raise ValueError(f"Unknown reward function: {self.reward_function}")

        # Add Magintude of the Drone's Speed
        drone_speed = numpy.linalg.norm(self.data.qvel[:3])
        reward += self.speed_factor * drone_speed

        # Read the sensor value
        made_contact = 0
        touch_reported = False
        # print(self.data.sensor(self.fly_sensor_id).data)
        sensor_reading = self.data.sensor(self.fly_sensor_id).data[0]

        # A positive value means the sensor is being touched
        if (sensor_reading > 0 and not touch_reported):
            # print(f"Drone touched the sensor! | Senor Force {sensor_reading:.3f} | Distance: {distance_to_target:.3f}")
            touch_reported = True
            reward += self.contact_bonus # Add a positive reward for touching the sensor
        elif distance_to_target <= self.sphere_size:
            made_contact = 1
            reward += self.contact_bonus # Add a positive reward for touching the sensor
            rounded_current_point = round_tuple_elements(self.current_point)
            if(self.total_contacts > 0 and rounded_current_point == (0,0,self.track_height)):
                self.laps += 1
                terminated = True
                reward += self.complete_bonus

            self._update_track_position()
            self.total_contacts += 1
            self.steps_without_contact = 0
        else:
            self.steps_without_contact += 1
            if self.steps_without_contact > self.terminate_without_contact:
                reward = self.out_of_bounds_penalty
                terminated = True

        if bool(distance_to_target > self.max_distance or drone_pos[2] < 0.025):
            reward = self.no_contact_penalty
            terminated = True
        elif self.step_number > self.episode_length:
            truncated = True

        # Penalized for taking too long
        if not terminated and not truncated:
            reward -= self.time_penalty

        # Update the total reward
        self.total_reward += reward

        observation = self._get_obs()

        info = {'distance_to_target': distance_to_target,
                "touch_reported": touch_reported,
                "sensor_reading": sensor_reading,
                "made_contact": made_contact,
                "total_reward": self.total_reward,
                "total_contacts": self.total_contacts,
                "drone_speed": drone_speed,
                "steps_without_contact": self.steps_without_contact,
                "laps": self.laps}

        return observation, reward, terminated, truncated, info

    def reset(self, seed=None, options=None):
        """Resets the environment to an initial state."""
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        noise = self.np_random.uniform(low=-0.1, high=0.1, size=3)
        self.data.qpos[:3] += noise

        # Make sure that the drone doesn't start too low
        self.data.qpos[2] = max(0.075, self.data.qpos[2])
        self.model.site_pos[self.target_pos_id] = self.original_site_pos.copy()
        self.target_pos = self.model.site_pos[self.target_pos_id]
        self.current_point = (0,0, self.track_height)
        self.next_point = get_next_clockwise_point(self.current_point, self.track_points)
        self.total_contacts = 0
        self.step_number = 0
        self.steps_without_contact = 0
        self.total_reward = 0
        self.laps = 0
        return self._get_obs(), {}

    # define what should happen when the model is reset (at the beginning of each episode)
    def reset_model(self, seed=None):
        self.step_number = 0
        self.total_reward = 0
        self.total_contacts = 0
        self.laps = 0
        self.steps_without_contact = 0

        self.current_point = (0,0, self.track_height)
        self.next_point = get_next_clockwise_point(self.current_point, self.track_points)

        # for example, noise is added to positions and velocities
        qpos = self.init_qpos + self.np_random.uniform(
            size=self.model.nq, low=-0.01, high=0.01
        )

        # Make sure that the drone doesn't start too low
        qpos[2] =  max(0.075, qpos[2])

        qvel = self.init_qvel + self.np_random.uniform(
            size=self.model.nv, low=-0.01, high=0.01
        )
        self.set_state(qpos, qvel)
        return self._get_obs()

In [ ]:
env = DroneHoverEnv(env_config)
print("Observation Space Size: ", env.observation_space.shape)
print('Actions Space: ', env.action_space)
env.close()

In [ ]:
def make_env():
  env = DroneHoverEnv(render_mode="rgb_array", env_config=env_config)
  check_env(env)
  return env

In [ ]:
# Create Training environment
env = make_vec_env(make_env,
                   n_envs=model_config["n_envs"],
                   monitor_dir=os.path.join(model_folder_path, "monitor"))

normalized_env = VecNormalize(env,
                              training=True,
                              norm_obs=True,
                              norm_reward=True)


# Create Evaluation environment
env_val = make_vec_env(make_env, n_envs=1)
normalized_env_val = VecNormalize(env_val,
                                  training=False,
                                  norm_obs=True,
                                  norm_reward=True)

checkpoint_callback = CheckpointCallback(
  save_freq=model_config["eval_freq"],
  save_path=os.path.join(model_folder_path, "checkpoints"),
  name_prefix="rl_model",
  save_replay_buffer=False,
  save_vecnormalize=True,
)

reformat_callback = ReformatEvalCallback(save_path=model_folder_path,
                                         eval_file=os.path.join(model_folder_path, "evaluations.npz"),
                                         save_freq=model_config["eval_freq"],
                                         csv_file_name="SummaryEvaluations")

vec_normalize_callback = VecNormalizeSaveCallback(save_path=model_folder_path)

eval_callback = EvalCallback(normalized_env_val,
                             best_model_save_path=model_folder_path,
                             log_path=model_folder_path,
                             render=False,
                             deterministic=True,
                             callback_on_new_best=vec_normalize_callback,
                             n_eval_episodes=model_config["n_eval_episodes"],
                             eval_freq=model_config["eval_freq"])

# Create the callback list
callbackList = CallbackList([checkpoint_callback,
                             eval_callback,
                             reformat_callback])

# learning with tensorboard logging and saving model
model = SAC("MlpPolicy",
            normalized_env,
            verbose=0,
            tensorboard_log=os.path.join(log_dir, "tensorboard"))

model.learn(total_timesteps=model_config["total_timesteps"],
            callback=callbackList,
            progress_bar=False)

# Save the model
model.save(os.path.join(model_folder_path, "final_model"))

mean_reward, std_reward = evaluate_policy(model,
                                          normalized_env_val,
                                          deterministic=True,
                                          n_eval_episodes=model_config["n_eval_episodes"])

print(f"Final Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

normalized_env.save(os.path.join(model_folder_path, "final_normalized_env.pkl"))
normalized_env_val.save(os.path.join(model_folder_path, "final_normalized_env_val.pkl"))

env.close()
env_val.close()
normalized_env.close()
normalized_env_val.close()

In [ ]:
# # Get model path from last training job (uncomment if training job interrupted)
# # List all entries in the directory
# entries = os.listdir(training_data_path)

# # Filter out only directories
# folders = [entry for entry in entries if os.path.isdir(os.path.join(training_data_path, entry))]

# # Sort the folders alphabetically
# folders.sort()

# # Get the last folder
# model_folder_path = os.path.join(training_data_path, folders[-1])
# print(model_folder_path)

In [ ]:
# Create Evaluation environment
env_val = make_vec_env(make_env, n_envs=1, seed=0)
normalized_env_val = VecNormalize.load(os.path.join(model_folder_path, "best_model_vec_normalize.pkl"),
                                       venv=env_val)

# Load the best model
best_model_path = os.path.join(model_folder_path, "best_model")
best_model = SAC.load(best_model_path, env=normalized_env_val)

mean_reward, std_reward = evaluate_policy(best_model,
                                          normalized_env_val,
                                          deterministic=True,
                                          n_eval_episodes=model_config["n_eval_episodes"])

print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

# Close the environment
env_val.close()
normalized_env_val.close()

In [ ]:
# Record video of the best model
video_length = 10_000
video_folder_path = os.path.join(model_folder_path, "videos")
os.makedirs(video_folder_path, exist_ok=True)
for i in range(notebook_config["video_playback"]):
    # Create Evaluation environment
    env_val = make_vec_env(make_env, n_envs=1)
    normalized_env_val = VecNormalize.load(os.path.join(model_folder_path, "best_model_vec_normalize.pkl"),
                                        venv=env_val)

    # Load the best model
    best_model_path = os.path.join(model_folder_path, "best_model")
    best_model = SAC.load(best_model_path, env=normalized_env_val)
    best_model_file_name = f"best_model_{name_prefix}_{i}"
    env = VecVideoRecorder(normalized_env_val, video_folder_path,
                           video_length=video_length,
                           record_video_trigger=lambda x: x == 0,
                           name_prefix=best_model_file_name)

    session_length = 0
    total_reward = 0
    obs_norm = env.reset()
    best_csv_file_name = os.path.join(video_folder_path, f"{best_model_file_name}.csv")
    with open(best_csv_file_name, 'w') as csvfile:
      csv_writer = csv.writer(csvfile, delimiter=',')
      csv_writer.writerow(csv_header)
      for _ in range(video_length):
          action, _states = best_model.predict(obs_norm, deterministic=True)
          obs_norm, rewards_norm, done, info = env.step(action)
          obs = env.get_original_obs()
          rewards = env.get_original_reward()
          total_reward += rewards
          row_data = numpy.concatenate([[int(session_length),
                                         obs[0][0],
                                         obs[0][1],
                                         obs[0][2],
                                         obs[0][-3],
                                         obs[0][-2],
                                         obs[0][-1],
                                         rewards[0],
                                         rewards_norm[0],
                                         total_reward[0],
                                         info[0]["touch_reported"],
                                         info[0]["sensor_reading"],
                                         info[0]["made_contact"],
                                         info[0]["laps"],
                                         info[0]["distance_to_target"],
                                         info[0]["drone_speed"],
                                         info[0]["steps_without_contact"],
                                         info[0]["total_contacts"],
                                         done[0]]])
          row_data = numpy.round(row_data, decimals=4)
          csv_writer.writerow(row_data)
          env.render()
          session_length += 1
          if done:
              break

    print(f"Total reward for playback {i+1}: {total_reward[0]:.2f}")

    # Close the environment
    env.close()
    env_val.close()
    normalized_env_val.close()

In [ ]:
# Load the evaluations.npz file
data = numpy.load(os.path.join(model_folder_path, "evaluations.npz"))

# Extract the relevant data
timesteps = data['timesteps']
results = data['results']

# Calculate the mean and standard deviation of the results
mean_results = numpy.mean(results, axis=1)
std_results = numpy.std(results, axis=1)

# Plot the results
matplotlib.pyplot.figure()
matplotlib.pyplot.plot(timesteps, mean_results)
matplotlib.pyplot.fill_between(timesteps,
                               mean_results - std_results,
                               mean_results + std_results,
                               alpha=0.3)

matplotlib.pyplot.xlabel('Timesteps')
matplotlib.pyplot.ylabel('Mean Reward')
matplotlib.pyplot.title(f"{rl_type} Performance on {env_str}")
matplotlib.pyplot.savefig(os.path.join(model_folder_path, f"{rl_type}_{env_str}_performance.png"))
matplotlib.pyplot.show()

In [ ]:
# Create version execution
notebook_name = "notebook.ipynb"
%notebook -e $notebook_name

In [ ]:
source_file = os.path.join(notebook_name)
destination_file = os.path.join(model_folder_path, notebook_name)

try:
    shutil.copyfile(notebook_name, destination_file)
    print(f"File '{source_file}' copied to '{destination_file}' successfully.")
except FileNotFoundError:
    print(f"Error: Source file '{source_file}' not found.")
except shutil.SameFileError:
    print(f"Error: Source and destination are the same file.")
except Exception as e:
    print(f"An error occurred: {e}")